# SQL-Crashkurs (ca. 1,5–2 h)

**Ziel:** Danach kannst du ehrlich „SQL (Grundkenntnisse)" in den CV schreiben – also Daten abfragen, filtern, aggregieren, Tabellen verknüpfen (JOIN) und Daten ändern.

**Szenario:** Eine kleine Datenbank eines THz-Messlabors: Mitarbeitende, Projekte, Proben und Messungen.

| Teil | Thema | Zeit |
|---|---|---|
| 0 | Setup | 5 min |
| 1 | SELECT, WHERE, ORDER BY | 20 min |
| 2 | Aggregation: COUNT, AVG, GROUP BY, HAVING | 20 min |
| 3 | JOINs | 25 min |
| 4 | Daten ändern: INSERT, UPDATE, DELETE | 10 min |
| 5 | Unterabfragen, CASE, WITH | 15 min |
| 6 | Abschlussaufgaben (ohne Hinweise) | 15 min |
| 7 | SQL + pandas | 5 min |

**So arbeitest du:** Beispielzellen ausführen und verstehen, dann die Aufgaben lösen. Bei jeder Aufgabe steht die erwartete Zeilenzahl als Kontrolle. Lösungen stehen in `SQL_Crashkurs_Loesungen.ipynb` – erst reinschauen, wenn du 5 Minuten festhängst.

## 0 · Setup

1. In VS Code die Extensions **Python** und **Jupyter** (beide von Microsoft) installieren.
2. Im Terminal: `pip install pandas` (SQLite ist in Python schon eingebaut, keine Datenbank-Installation nötig).
3. Oben rechts im Notebook **Select Kernel** → deine Python-Installation wählen.
4. Die nächste Zelle ausführen (`Shift+Enter`). Sie legt `labor.db` an und definiert zwei Helfer:
   - `q("SELECT ...")` – Abfrage ausführen, Ergebnis als Tabelle
   - `run("UPDATE ...")` – Daten ändern

**Reset:** Diese Zelle erneut ausführen setzt die Datenbank komplett zurück.

Optional: Mit der Extension **SQLite Viewer** kannst du `labor.db` per Klick im Explorer ansehen.

In [ ]:
import sqlite3, random, os
import pandas as pd

DB = "labor.db"
if os.path.exists(DB):
    os.remove(DB)              # bei jedem Ausführen frisch aufsetzen = Reset
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON")

con.executescript("""
CREATE TABLE mitarbeiter (
    id        INTEGER PRIMARY KEY,
    name      TEXT NOT NULL,
    rolle     TEXT,
    eintritt  DATE
);
CREATE TABLE projekte (
    id          INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    kunde       TEXT,
    budget_eur  INTEGER,
    start       DATE
);
CREATE TABLE proben (
    id          INTEGER PRIMARY KEY,
    bezeichnung TEXT NOT NULL,
    material    TEXT,
    dicke_um    REAL,
    projekt_id  INTEGER REFERENCES projekte(id)
);
CREATE TABLE messungen (
    id              INTEGER PRIMARY KEY,
    probe_id        INTEGER REFERENCES proben(id),
    mitarbeiter_id  INTEGER REFERENCES mitarbeiter(id),
    datum           DATE,
    methode         TEXT,
    frequenz_thz    REAL,
    snr_db          REAL,
    status          TEXT
);
""")

random.seed(42)
mitarbeiter = [
    (1, "Anna Berger",  "Doktorandin",  "2021-04-01"),
    (2, "Jonas Kraft",  "Postdoc",      "2019-10-01"),
    (3, "Mira Yilmaz",  "Hilfskraft",   "2023-03-15"),
    (4, "Tim Hoffmann", "Hilfskraft",   "2024-01-08"),
    (5, "Lea Schulz",   "Gruppenleitung","2015-06-01"),
    (6, "Paul Neumann", "Masterand",    "2025-02-01"),
]
projekte = [
    (1, "Lackschichtdicke",   "AutoCoat GmbH",    85000, "2024-02-01"),
    (2, "Feuchte in Blattproben", "AgriSense AG", 42000, "2024-09-01"),
    (3, "Polymer-Charakterisierung", "PolyTec KG", 120000, "2025-01-15"),
    (4, "Halbleiter-QS",      "ChipLine GmbH",   250000, "2025-05-01"),
    (5, "Bildgebung Demo",    None,               15000, "2025-11-01"),
]
materialien = ["Silizium", "Quarz", "PTFE", "HDPE", "Polyimid", "Saphir"]
proben = []
for i in range(1, 25):
    mat = random.choice(materialien)
    dicke = round(random.uniform(50, 2000), 1)
    pid = random.choice([1, 1, 2, 3, 3, 4, 4])
    proben.append((i, f"P-{i:03d}", mat, dicke, pid))
proben.append((25, "P-025", "Silizium", None, 4))       # Dicke unbekannt (NULL)
for i in (5, 11, 19):                                  # drei Proben ohne Projekt
    proben[i - 1] = proben[i - 1][:4] + (None,)

messungen = []
for i in range(1, 241):
    probe = random.randint(1, 22)                       # P-023..P-025 nie gemessen
    ma = random.choice([1, 1, 2, 2, 3, 3, 3, 4, 4, 6])  # Lea misst nie selbst
    monat = random.randint(1, 12)
    jahr = random.choice([2024, 2025])
    datum = f"{jahr}-{monat:02d}-{random.randint(1, 28):02d}"
    methode = random.choice(["TDS", "TDS", "FDS"])
    freq = round(random.uniform(0.1, 3.0), 2)
    snr = round(random.gauss(45, 10), 1)
    status = "fehlerhaft" if snr < 30 else "ok"
    messungen.append((i, probe, ma, datum, methode, freq, snr, status))

con.executemany("INSERT INTO mitarbeiter VALUES (?,?,?,?)", mitarbeiter)
con.executemany("INSERT INTO projekte VALUES (?,?,?,?,?)", projekte)
con.executemany("INSERT INTO proben VALUES (?,?,?,?,?)", proben)
con.executemany("INSERT INTO messungen VALUES (?,?,?,?,?,?,?,?)", messungen)
con.commit()

def _leer(sql):
    if not "".join(l for l in sql.splitlines() if not l.strip().startswith("--")).strip():
        print("Noch keine Abfrage – ersetze die Kommentarzeile durch dein SQL.")
        return True
    return False

def q(sql, params=()):
    """SELECT ausführen und Ergebnis als Tabelle anzeigen."""
    if _leer(sql): return
    return pd.read_sql_query(sql, con, params=params)

def run(sql):
    """INSERT/UPDATE/DELETE/CREATE ausführen und speichern."""
    if _leer(sql): return
    cur = con.execute(sql)
    con.commit()
    print(f"OK – {cur.rowcount} Zeile(n) betroffen")

print("Datenbank bereit:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")])

### Die Datenbank

```
mitarbeiter (id, name, rolle, eintritt)
projekte    (id, name, kunde, budget_eur, start)
proben      (id, bezeichnung, material, dicke_um, projekt_id → projekte.id)
messungen   (id, probe_id → proben.id, mitarbeiter_id → mitarbeiter.id,
             datum, methode, frequenz_thz, snr_db, status)
```

- **Primärschlüssel** (`id`): identifiziert eine Zeile eindeutig.
- **Fremdschlüssel** (z. B. `proben.projekt_id`): verweist auf den Primärschlüssel einer anderen Tabelle. Darüber werden Tabellen später per JOIN verknüpft.
- `NULL` = kein Wert vorhanden (nicht 0, nicht leer).

## 1 · SELECT, WHERE, ORDER BY

Grundform einer Abfrage:

```sql
SELECT spalten
FROM tabelle
WHERE bedingung
ORDER BY spalte [ASC|DESC]
LIMIT n;
```

SQL-Schlüsselwörter sind nicht case-sensitiv; Großschreibung ist nur Konvention. Text steht in **einfachen** Anführungszeichen: `'PTFE'`.

In [ ]:
q("""
SELECT * FROM proben LIMIT 5
""")

In [ ]:
q("""
SELECT bezeichnung, material, dicke_um FROM proben WHERE material = 'Quarz'
""")

**Bedingungen kombinieren:** `AND`, `OR`, `NOT` · Vergleich: `=  <>  <  >  <=  >=`

Nützliche Operatoren:
- `IN ('a', 'b')` – Wert in Liste
- `BETWEEN 100 AND 500` – Bereich (inklusive Grenzen)
- `LIKE 'P-00%'` – Muster: `%` = beliebig viele Zeichen, `_` = genau eins
- `IS NULL` / `IS NOT NULL` – **nie** `= NULL` schreiben, das liefert immer nichts

In [ ]:
q("""
SELECT bezeichnung, material, dicke_um
FROM proben
WHERE material IN ('PTFE', 'HDPE') AND dicke_um > 500
ORDER BY dicke_um DESC
""")

`DISTINCT` entfernt Duplikate, `AS` vergibt einen Spaltennamen (Alias). Rechnen geht direkt in SELECT:

In [ ]:
q("""
SELECT DISTINCT material FROM proben ORDER BY material
""")

In [ ]:
q("""
SELECT bezeichnung, dicke_um / 1000.0 AS dicke_mm FROM proben LIMIT 5
""")

**Aufgabe 1.1** – Alle Mitarbeitenden mit der Rolle `Hilfskraft` (alle Spalten).

*Kontrolle: 2 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 1.2** – Die 5 Messungen mit dem höchsten Signal-Rausch-Verhältnis (`snr_db`). Zeige `id`, `datum`, `methode`, `snr_db`.

*Kontrolle: 5 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 1.3** – Alle FDS-Messungen aus dem Jahr 2025 mit Status `ok`. Daten sind Text im Format `JJJJ-MM-TT` – `BETWEEN '2025-01-01' AND '2025-12-31'` funktioniert deshalb.

*Kontrolle: 30 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 1.4** – Alle Proben, die keinem Projekt zugeordnet sind.

*Kontrolle: 3 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 1.5** – Bezeichnung und Dicke in mm (Alias `dicke_mm`) aller Proben zwischen 0,2 und 0,8 mm, aufsteigend nach Dicke sortiert. Achtung: `dicke_um` ist in µm.

*Kontrolle: 11 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

## 2 · Aggregation: COUNT, AVG, GROUP BY, HAVING

Aggregatfunktionen fassen viele Zeilen zu einem Wert zusammen: `COUNT(*)`, `SUM()`, `AVG()`, `MIN()`, `MAX()`. `ROUND(x, 1)` rundet.

In [ ]:
q("""
SELECT COUNT(*) AS anzahl, ROUND(AVG(snr_db), 1) AS snr_mittel, MAX(frequenz_thz) AS f_max FROM messungen
""")

**GROUP BY** bildet Gruppen und berechnet das Aggregat **pro Gruppe**. Faustregel: Jede Spalte im SELECT ist entweder in GROUP BY oder steckt in einer Aggregatfunktion.

In [ ]:
q("""
SELECT methode, COUNT(*) AS anzahl, ROUND(AVG(snr_db), 1) AS snr_mittel
FROM messungen
GROUP BY methode
""")

**WHERE vs. HAVING:**
- `WHERE` filtert **Zeilen vor** dem Gruppieren.
- `HAVING` filtert **Gruppen nach** dem Gruppieren (kann daher Aggregate nutzen).

Hinweis: `COUNT(spalte)` zählt nur Nicht-NULL-Werte, `COUNT(*)` zählt alle Zeilen.

In [ ]:
q("""
SELECT probe_id, COUNT(*) AS anzahl
FROM messungen
WHERE status = 'ok'
GROUP BY probe_id
HAVING COUNT(*) >= 12
ORDER BY anzahl DESC
""")

**Logische Ausführungsreihenfolge** (erklärt viele Fehlermeldungen):

`FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`

Deshalb kann `WHERE` einen in SELECT vergebenen Alias nicht kennen, `ORDER BY` aber schon.

**Aufgabe 2.1** – Wie viele Proben gibt es pro Material? Absteigend nach Anzahl sortiert.

*Kontrolle: 6 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 2.2** – Fehlerquote: Anzahl Messungen pro `status`.

*Kontrolle: 2 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 2.3** – Mittleres SNR (gerundet auf 1 Nachkommastelle) pro `mitarbeiter_id`, nur für Messungen mit Status `ok`.

*Kontrolle: 5 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 2.4** – Welche Materialien haben eine mittlere Probendicke über 1000 µm? Zeige Material und mittlere Dicke.

*Kontrolle: 1 Zeile*

In [ ]:
q("""
-- deine Lösung
""")

## 3 · JOINs – Tabellen verknüpfen

Bisher steht in `messungen` nur `mitarbeiter_id = 3`. Den Namen holt ein JOIN über den Fremdschlüssel:

```sql
SELECT ...
FROM tabelle_a AS a
JOIN tabelle_b AS b ON a.fremdschluessel = b.id
```

- **INNER JOIN** (kurz `JOIN`): nur Zeilen, die in **beiden** Tabellen einen Partner haben.
- **LEFT JOIN**: **alle** Zeilen der linken Tabelle; fehlt rechts ein Partner, stehen dort `NULL`-Werte.

Tabellen-Aliase (`m`, `p`) sparen Tipparbeit und sind bei gleichnamigen Spalten (`id`, `name`) Pflicht.

In [ ]:
q("""
SELECT m.id, m.datum, ma.name, m.snr_db
FROM messungen AS m
JOIN mitarbeiter AS ma ON m.mitarbeiter_id = ma.id
LIMIT 5
""")

Unterschied INNER vs. LEFT – Proben mit ihrem Projekt. Vergleiche die Zeilenzahl:

In [ ]:
q("""
SELECT p.bezeichnung, pr.name AS projekt
FROM proben AS p
LEFT JOIN projekte AS pr ON p.projekt_id = pr.id
""")

Mehrere JOINs hintereinander, kombiniert mit GROUP BY:

In [ ]:
q("""
SELECT pr.name AS projekt, COUNT(m.id) AS messungen
FROM projekte AS pr
JOIN proben AS p     ON p.projekt_id = pr.id
JOIN messungen AS m  ON m.probe_id   = p.id
GROUP BY pr.name
ORDER BY messungen DESC
""")

**Aufgabe 3.1** – Alle Messungen mit Status `fehlerhaft`: Datum, Name der messenden Person, Probenbezeichnung, SNR. Nach Datum sortiert.

*Kontrolle: 15 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 3.2** – Anzahl Messungen pro Person (Name), absteigend. **Alle** Mitarbeitenden sollen erscheinen – auch wer nie gemessen hat (dann 0). Tipp: LEFT JOIN und `COUNT(m.id)` statt `COUNT(*)`.

*Kontrolle: 6 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 3.3** – Welche Proben wurden **nie** gemessen? Tipp: LEFT JOIN und dann auf `NULL` auf der rechten Seite filtern.

*Kontrolle: 3 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 3.4** – Pro Kunde: Anzahl Proben und mittlere Probendicke. Nur Projekte mit eingetragenem Kunden.

*Kontrolle: 4 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

## 4 · Daten ändern: INSERT, UPDATE, DELETE

```sql
INSERT INTO tabelle (spalte1, spalte2) VALUES ('wert1', 2);
UPDATE tabelle SET spalte = wert WHERE bedingung;
DELETE FROM tabelle WHERE bedingung;
```

**Wichtigste Regel:** `UPDATE` und `DELETE` **immer mit WHERE**. Ohne WHERE trifft es jede Zeile der Tabelle. Profi-Gewohnheit: erst mit `SELECT ... WHERE ...` prüfen, welche Zeilen getroffen werden, dann ausführen.

Hier wird `run(...)` statt `q(...)` verwendet. Falls etwas schiefgeht: Setup-Zelle erneut ausführen.

In [ ]:
run("""
INSERT INTO mitarbeiter (id, name, rolle, eintritt)
VALUES (7, 'Florian Lemke', 'Hilfskraft', '2018-09-01')
""")

**Aufgabe 4.1** – Lege ein neues Projekt an: `id` 6, Name `Astro-Detektoren`, Kunde `SpaceOptics GmbH`, Budget 60000, Start `2026-10-01`. Kontrolle danach mit `q("SELECT * FROM projekte")`.

*Kontrolle: 1 Zeile betroffen*

In [ ]:
run("""
-- deine Lösung
""")

**Aufgabe 4.2** – Ordne alle Proben ohne Projekt dem neuen Projekt 6 zu.

*Kontrolle: 3 Zeilen betroffen*

In [ ]:
run("""
-- deine Lösung
""")

**Aufgabe 4.3** – Lösche alle fehlerhaften Messungen mit einem SNR unter 20 dB. Prüfe vorher mit SELECT, wie viele es sind.

*Kontrolle: 2 Zeilen betroffen*

In [ ]:
run("""
-- deine Lösung
""")

Zum Wissen (nicht üben): `CREATE TABLE` hast du in der Setup-Zelle gesehen. `ALTER TABLE` ändert Tabellenstrukturen, `DROP TABLE` löscht Tabellen komplett.

**Jetzt die Setup-Zelle erneut ausführen**, damit Teil 5 und 6 mit den Originaldaten arbeiten.

## 5 · Unterabfragen, CASE, WITH

**Unterabfrage (Subquery):** eine Abfrage in der Abfrage, z. B. um mit einem berechneten Wert zu vergleichen.

In [ ]:
q("""
SELECT id, snr_db
FROM messungen
WHERE snr_db > (SELECT AVG(snr_db) FROM messungen) + 15
""")

**CASE WHEN** – Fallunterscheidung, wie if/elif/else:

In [ ]:
q("""
SELECT bezeichnung, dicke_um,
       CASE
           WHEN dicke_um < 500  THEN 'dünn'
           WHEN dicke_um < 1500 THEN 'mittel'
           ELSE 'dick'
       END AS klasse
FROM proben
LIMIT 8
""")

**WITH (CTE, Common Table Expression)** – benannte Zwischentabelle. Macht lange Abfragen lesbar, ähnlich wie eine Zwischenvariable in Python:

In [ ]:
q("""
WITH pro_probe AS (
    SELECT probe_id, COUNT(*) AS n, AVG(snr_db) AS snr
    FROM messungen
    GROUP BY probe_id
)
SELECT p.bezeichnung, p.material, pp.n, ROUND(pp.snr, 1) AS snr_mittel
FROM pro_probe AS pp
JOIN proben AS p ON p.id = pp.probe_id
ORDER BY pp.snr DESC
LIMIT 5
""")

**Datum in SQLite:** `strftime('%Y', datum)` liefert das Jahr, `'%m'` den Monat. (In PostgreSQL wäre das `EXTRACT(YEAR FROM datum)` – Datumsfunktionen unterscheiden sich je nach Datenbank am stärksten.)

**Aufgabe 5.1** – Alle Proben, die dicker sind als die durchschnittliche Probe. Zeige Bezeichnung und Dicke.

*Kontrolle: 11 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 5.2** – Anzahl Messungen pro Jahr und Methode (Spalten `jahr`, `methode`, `anzahl`).

*Kontrolle: 4 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 5.3** – Pro Methode: Anzahl Messungen gesamt und Fehlerquote in Prozent (1 Nachkommastelle). Tipp: `SUM(CASE WHEN status = 'fehlerhaft' THEN 1 ELSE 0 END)` zählt die Fehler. Achtung Ganzzahldivision: mit `100.0` multiplizieren.

*Kontrolle: 2 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

## 6 · Abschlussaufgaben

Ohne Tipps. Wenn du diese drei selbstständig lösen kannst, ist „SQL (Grundkenntnisse)" im CV gedeckt.

**Aufgabe 6.1** – Projektbericht: Für jedes Projekt Name, Kunde, Budget, Anzahl Messungen und mittleres SNR. Nur Projekte mit mindestens 30 Messungen, nach Anzahl absteigend.

*Kontrolle: 2 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 6.2** – Für jede Person (Name): Datum der ersten und der letzten eigenen Messung. Nur Personen, die gemessen haben.

*Kontrolle: 5 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

**Aufgabe 6.3** – Qualitätscheck: Welche Materialien haben bei TDS-Messungen ein mittleres SNR unter dem mittleren SNR **aller** TDS-Messungen? Zeige Material und mittleres SNR.

*Kontrolle: 2 Zeilen*

In [ ]:
q("""
-- deine Lösung
""")

## 7 · SQL + pandas

In der Praxis (Data Science, Messdaten) kombinierst du beides: SQL holt und filtert die Daten aus der Datenbank, pandas/matplotlib übernehmen Analyse und Plot.

**Parameter immer per `?` übergeben**, nie per f-String in den SQL-Text einbauen – sonst SQL-Injection-Gefahr (ein Klassiker in Interviews).

In [ ]:
material = "PTFE"
df = q("""
    SELECT m.frequenz_thz, m.snr_db
    FROM messungen AS m
    JOIN proben AS p ON p.id = m.probe_id
    WHERE p.material = ?
""", params=(material,))

df.plot.scatter(x="frequenz_thz", y="snr_db", title=f"SNR vs. Frequenz – {material}");

In [ ]:
# Umgekehrt: DataFrame als Tabelle in die Datenbank schreiben
neu = pd.DataFrame({"material": ["Quarz", "PTFE"], "brechungsindex": [1.95, 1.43]})
neu.to_sql("materialdaten", con, if_exists="replace", index=False)
q("SELECT * FROM materialdaten")

## Kurzreferenz

| Befehl | Zweck |
|---|---|
| `SELECT … FROM … WHERE …` | Abfragen und filtern |
| `ORDER BY … DESC`, `LIMIT n` | Sortieren, begrenzen |
| `COUNT/SUM/AVG/MIN/MAX` + `GROUP BY` | Aggregieren pro Gruppe |
| `HAVING` | Gruppen filtern |
| `JOIN … ON …` / `LEFT JOIN` | Tabellen verknüpfen |
| `CASE WHEN … THEN … END` | Fallunterscheidung |
| `WITH name AS (…)` | Zwischentabelle |
| `INSERT` / `UPDATE` / `DELETE` | Daten ändern (immer mit WHERE!) |

**Dialekte:** Grundsyntax ist überall gleich (SQLite, PostgreSQL, MySQL, Microsoft SQL Server). Unterschiede v. a. bei Datumsfunktionen und `LIMIT` (SQL Server: `SELECT TOP n`).

**Weiterüben:** sqlbolt.com (interaktiv, kostenlos), danach SQL-Aufgaben auf HackerRank oder LeetCode (Stufe „Easy").